# SPY Daily Price Acquisition and Storage Pipeline

This Stage 04–05 pipeline acquires real adjusted daily OHLCV data through `yfinance`, validates a stable schema, and reconciles CSV and Parquet copies.

## Setup

In [1]:
from pathlib import Path
import os, sys

if Path.cwd().name == 'notebooks':
    os.chdir('..')
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('working from:', ROOT.name)

working from: project


## Parameters and source

- Source: Yahoo Finance historical market data via the `yfinance` client
- Instrument: SPY (broad U.S. equity-market ETF proxy)
- Frequency: daily, with `auto_adjust=True`
- Source page: https://finance.yahoo.com/quote/SPY/history/

The open-ended end date intentionally refreshes to the latest available trading day.

In [2]:
from src.config import load_env

load_env()
SYMBOL = 'SPY'
START_DATE = '2015-01-01'
END_DATE = None
print({'symbol': SYMBOL, 'start': START_DATE, 'end': END_DATE})

{'symbol': 'SPY', 'start': '2015-01-01', 'end': None}


## Acquire and validate

In [3]:
from src.ingestion import download_daily_prices, validate_market_data

prices = download_daily_prices(SYMBOL, START_DATE, END_DATE)
validate_market_data(prices)
print('rows:', len(prices))
print('date range:', prices['date'].min().date(), 'to', prices['date'].max().date())
print('columns:', prices.columns.tolist())
print('missing values:', prices.isna().sum().to_dict())
print('duplicate dates:', int(prices['date'].duplicated().sum()))
prices.head()

rows: 2922
date range: 2015-01-02 to 2026-08-17
columns: ['date', 'open', 'high', 'low', 'close', 'volume']
missing values: {'date': 0, 'open': 0, 'high': 0, 'low': 0, 'close': 0, 'volume': 0}
duplicate dates: 0


,date,open,high,low,close,volume
0,2015-01-02,170.472573,170.885580,168.655335,169.687851,121465900
1,2015-01-05,168.647051,168.812251,166.317701,166.623322,169632600
2,2015-01-06,166.928950,167.449342,164.260931,165.053909,209151400
3,2015-01-07,166.375491,167.449310,165.929449,167.110641,125346700
4,2015-01-08,168.514901,170.290837,168.498390,170.076065,147217800


### Validation logic

The reusable validator requires at least 20 rows, exact ordered columns, unique increasing dates, nonmissing positive OHLC prices, and nonnegative volume. A download that returns successfully but violates any rule fails the pipeline.

## Store and reconcile

In [4]:
from src.storage import resolve_data_dir, save_raw_snapshot, verify_storage_round_trip

data_dir = resolve_data_dir(ROOT)
csv_path, parquet_path = save_raw_snapshot(prices, SYMBOL, data_dir)
verify_storage_round_trip(prices, csv_path, parquet_path)
for path in (csv_path, parquet_path):
    print(path.relative_to(ROOT), f'{path.stat().st_size:,} bytes')
print('CSV and Parquet both reconcile with the validated source frame.')

data/raw/spy_daily.csv 273,129 bytes
data/raw/spy_daily.parquet 151,666 bytes
CSV and Parquet both reconcile with the validated source frame.


## Checks and limitations

Yahoo Finance is a convenient course-scale source, not a contractual institutional feed. Availability, schema, adjustments, and historical observations may change. The raw snapshot is therefore validated and stored in two formats, but it should be re-acquired and compared before consequential use.

In [5]:
assert len(prices) > 1_000
assert prices['date'].is_monotonic_increasing and prices['date'].is_unique
assert (prices[['open', 'high', 'low', 'close']] > 0).all().all()
assert csv_path.is_file() and parquet_path.is_file()
print('All Stage 04–05 pipeline checks passed.')

All Stage 04–05 pipeline checks passed.
